# Original-feature unsupervised intrusion-detection experiment

This notebook evaluates K-Means, DBSCAN, Agglomerative Hierarchical Clustering, and Gaussian Mixture Models on the 41 original NFS features. Labels are excluded from preprocessing and clustering; they are used only after fitting for external evaluation and cluster-to-class mapping.

Run names follow `original_nfs_{algorithm}`, such as `original_nfs_k-means`. All parameters, sampled inputs, internal/external metrics, mappings, plots, metadata, and fitted artifacts are tracked in MLflow.

In [ ]:
import json
import platform
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import sklearn
from IPython.display import Markdown, display
from mlflow.models import infer_signature
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.manifold import TSNE
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, adjusted_rand_score,
    calinski_harabasz_score, classification_report, completeness_score,
    confusion_matrix, davies_bouldin_score, f1_score,
    homogeneity_score, normalized_mutual_info_score, precision_score,
    precision_recall_fscore_support, recall_score, silhouette_score,
    v_measure_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from configs.config import EXPERIMENT_NAME, RANDOM_STATE

TRACKING_DB = (PROJECT_ROOT / "Notebooks" / "mlflow.db").resolve()
TRACKING_URI = f"sqlite:///{TRACKING_DB.as_posix()}"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

CLUSTER_TRAIN_SIZE = 8_000
CLUSTER_TEST_SIZE = 4_000
CLUSTERING_PCA_COMPONENTS = 20
INTERNAL_METRIC_SAMPLE_SIZE = 3_000
TSNE_SAMPLE_SIZE = 2_000
TARGET_CLUSTER_COUNT = 2
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

In [ ]:
data_path = PROJECT_ROOT / "Data" / "Consolidated_df.csv"
df = pd.read_csv(data_path)
ORIGINAL_FEATURES = (
    "duration", "protocoltype", "service", "flag", "srcbytes",
    "dstbytes", "land", "wrongfragment", "urgent", "hot",
    "numfailedlogins", "loggedin", "numcompromised", "rootshell",
    "suattempted", "numroot", "numfilecreations", "numshells",
    "numaccessfiles", "numoutboundcmds", "ishostlogin",
    "isguestlogin", "count", "srvcount", "serrorrate",
    "srvserrorrate", "rerrorrate", "srvrerrorrate", "samesrvrate",
    "diffsrvrate", "srvdiffhostrate", "dsthostcount",
    "dsthostsrvcount", "dsthostsamesrvrate", "dsthostdiffsrvrate",
    "dsthostsamesrcportrate", "dsthostsrvdiffhostrate",
    "dsthostserrorrate", "dsthostsrvserrorrate",
    "dsthostrerrorrate", "dsthostsrvrerrorrate",
)
X = df[list(ORIGINAL_FEATURES)]
y = df["binary_target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
label_mapping = {"Normal": 0, "Attack": 1}
y_train_encoded = y_train.astype(str).map(label_mapping)
y_test_encoded = y_test.astype(str).map(label_mapping)

def stratified_sample(X_frame, y_series, sample_size, seed):
    if sample_size >= len(X_frame):
        return X_frame.copy(), y_series.copy()
    indices, _ = train_test_split(
        np.arange(len(X_frame)), train_size=sample_size,
        stratify=y_series, random_state=seed,
    )
    return X_frame.iloc[indices].copy(), y_series.iloc[indices].copy()

X_cluster_train, y_cluster_train = stratified_sample(
    X_train, y_train_encoded, CLUSTER_TRAIN_SIZE, RANDOM_STATE
)
X_cluster_test, y_cluster_test = stratified_sample(
    X_test, y_test_encoded, CLUSTER_TEST_SIZE, RANDOM_STATE + 1
)
categorical_features = X_cluster_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
numeric_features = [c for c in ORIGINAL_FEATURES if c not in categorical_features]
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
], remainder="drop", verbose_feature_names_out=False)
X_train_preprocessed = preprocessor.fit_transform(X_cluster_train)
X_test_preprocessed = preprocessor.transform(X_cluster_test)
pca_cluster = PCA(
    n_components=min(CLUSTERING_PCA_COMPONENTS, X_train_preprocessed.shape[1]),
    random_state=RANDOM_STATE,
)
X_train_cluster_space = pca_cluster.fit_transform(X_train_preprocessed)
X_test_cluster_space = pca_cluster.transform(X_test_preprocessed)
representation_pipeline = Pipeline([
    ("preprocessor", preprocessor), ("pca", pca_cluster),
])
print(
    f"Full split train={len(X_train)}, test={len(X_test)}; "
    f"clustering sample train={len(X_cluster_train)}, test={len(X_cluster_test)}; "
    f"cluster space={X_train_cluster_space.shape[1]} components; "
    f"explained variance={pca_cluster.explained_variance_ratio_.sum():.3f}"
)

In [ ]:
def fit_hungarian_mapping(cluster_ids, true_labels):
    clusters = np.unique(cluster_ids)
    classes = np.unique(true_labels)
    contingency = np.zeros((len(clusters), len(classes)), dtype=int)
    for i, cluster_id in enumerate(clusters):
        for j, class_id in enumerate(classes):
            contingency[i, j] = np.sum((cluster_ids == cluster_id) & (true_labels == class_id))
    row_indices, col_indices = linear_sum_assignment(-contingency)
    mapping = {int(clusters[i]): int(classes[j]) for i, j in zip(row_indices, col_indices)}
    global_majority = int(pd.Series(true_labels).mode().iloc[0])
    for cluster_id in clusters:
        if int(cluster_id) not in mapping:
            labels = true_labels[cluster_ids == cluster_id]
            mapping[int(cluster_id)] = (
                int(pd.Series(labels).mode().iloc[0]) if len(labels) else global_majority
            )
    return mapping, global_majority, contingency.tolist()

def apply_cluster_mapping(cluster_ids, mapping, default_label):
    return np.array([mapping.get(int(cluster_id), default_label) for cluster_id in cluster_ids])

def predict_nearest_centroid(X_values, train_values, train_clusters):
    cluster_ids = np.unique(train_clusters)
    centroids = np.vstack([train_values[train_clusters == c].mean(axis=0) for c in cluster_ids])
    distances = ((X_values[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    return cluster_ids[np.argmin(distances, axis=1)], centroids

def predict_dbscan(model, X_values):
    if len(model.core_sample_indices_) == 0:
        return np.full(len(X_values), -1, dtype=int)
    core_points = model.components_
    core_labels = model.labels_[model.core_sample_indices_]
    neighbors = NearestNeighbors(n_neighbors=1).fit(core_points)
    distances, indices = neighbors.kneighbors(X_values)
    predictions = core_labels[indices[:, 0]].astype(int)
    predictions[distances[:, 0] > model.eps] = -1
    return predictions

def safe_internal_metrics(X_values, clusters):
    unique_clusters = np.unique(clusters)
    if len(unique_clusters) < 2 or len(unique_clusters) >= len(clusters):
        return {"silhouette_score": np.nan, "davies_bouldin_index": np.nan, "calinski_harabasz_index": np.nan}
    sample_size = min(INTERNAL_METRIC_SAMPLE_SIZE, len(X_values))
    return {
        "silhouette_score": silhouette_score(
            X_values, clusters, sample_size=sample_size, random_state=RANDOM_STATE
        ),
        "davies_bouldin_index": davies_bouldin_score(X_values, clusters),
        "calinski_harabasz_index": calinski_harabasz_score(X_values, clusters),
    }

def all_metrics(X_values, clusters, true_labels, mapped_predictions):
    metrics = safe_internal_metrics(X_values, clusters)
    metrics.update({
        "adjusted_rand_index": adjusted_rand_score(true_labels, clusters),
        "normalized_mutual_information": normalized_mutual_info_score(true_labels, clusters),
        "homogeneity": homogeneity_score(true_labels, clusters),
        "completeness": completeness_score(true_labels, clusters),
        "v_measure": v_measure_score(true_labels, clusters),
        "mapped_accuracy": accuracy_score(true_labels, mapped_predictions),
        "mapped_precision": precision_score(true_labels, mapped_predictions, zero_division=0),
        "mapped_recall": recall_score(true_labels, mapped_predictions, zero_division=0),
        "mapped_f1": f1_score(true_labels, mapped_predictions, zero_division=0),
        "cluster_count": int(len(np.unique(clusters))),
        "noise_ratio": float(np.mean(clusters == -1)),
    })
    return metrics

def create_clusterer(algorithm_name, params):
    if algorithm_name == "k-means":
        return KMeans(random_state=RANDOM_STATE, **params)
    if algorithm_name == "dbscan":
        return DBSCAN(**params)
    if algorithm_name == "agglomerative-hierarchical":
        return AgglomerativeClustering(**params)
    if algorithm_name == "gaussian-mixture-model":
        return GaussianMixture(random_state=RANDOM_STATE, **params)
    raise KeyError(algorithm_name)

def fit_and_assign(algorithm_name, params, X_train_values, X_test_values):
    model = create_clusterer(algorithm_name, params)
    started = time.perf_counter()
    if algorithm_name in {"k-means", "gaussian-mixture-model"}:
        model.fit(X_train_values)
        train_clusters = model.predict(X_train_values)
        test_clusters = model.predict(X_test_values)
        assignment_metadata = {}
    else:
        train_clusters = model.fit_predict(X_train_values)
        if algorithm_name == "dbscan":
            test_clusters = predict_dbscan(model, X_test_values)
            assignment_metadata = {"test_assignment": "nearest_core_point_within_eps"}
        else:
            test_clusters, centroids = predict_nearest_centroid(
                X_test_values, X_train_values, train_clusters
            )
            assignment_metadata = {
                "test_assignment": "nearest_training_cluster_centroid",
                "centroids": centroids.tolist(),
            }
    return model, train_clusters, test_clusters, time.perf_counter() - started, assignment_metadata


## Fit and track the four clustering algorithms

DBSCAN's initial `eps` is estimated from the 90th percentile of training ten-nearest-neighbor distances. Agglomerative and DBSCAN test assignments use documented out-of-sample approximations because scikit-learn does not provide native `predict` methods for those estimators. Cluster-to-label mappings are learned from training labels using the Hungarian algorithm and then applied unchanged to test clusters.

In [1]:
neighbor_probe = NearestNeighbors(n_neighbors=10).fit(X_train_cluster_space)
neighbor_distances, _ = neighbor_probe.kneighbors(X_train_cluster_space)
dbscan_eps = float(np.quantile(neighbor_distances[:, -1], 0.90))
algorithm_parameters = {
    "k-means": {"n_clusters": TARGET_CLUSTER_COUNT, "n_init": 20, "max_iter": 500},
    "dbscan": {"eps": dbscan_eps, "min_samples": 10, "metric": "euclidean", "n_jobs": -1},
    "agglomerative-hierarchical": {"n_clusters": TARGET_CLUSTER_COUNT, "linkage": "ward"},
    "gaussian-mixture-model": {"n_components": TARGET_CLUSTER_COUNT, "covariance_type": "diag", "n_init": 5, "max_iter": 300},
}

visualization_size = min(TSNE_SAMPLE_SIZE, len(X_test_cluster_space))
visualization_indices, _ = train_test_split(
    np.arange(len(X_test_cluster_space)), train_size=visualization_size,
    stratify=y_cluster_test, random_state=RANDOM_STATE,
)
pca_visual = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(
    X_test_cluster_space[visualization_indices]
)
tsne_visual = TSNE(
    n_components=2, perplexity=30, init="pca", learning_rate="auto",
    max_iter=1000, random_state=RANDOM_STATE,
).fit_transform(X_test_cluster_space[visualization_indices])

train_tracking_df = X_cluster_train.copy()
train_tracking_df["binary_target"] = y_cluster_train.map({0: "Normal", 1: "Attack"})
test_tracking_df = X_cluster_test.copy()
test_tracking_df["binary_target"] = y_cluster_test.map({0: "Normal", 1: "Attack"})
train_dataset = mlflow.data.from_pandas(
    train_tracking_df, source=str(data_path.resolve()), targets="binary_target",
    name="original_nfs_unsupervised_train_sample",
)
test_dataset = mlflow.data.from_pandas(
    test_tracking_df, source=str(data_path.resolve()), targets="binary_target",
    name="original_nfs_unsupervised_test_sample",
)

results = []
fitted_runs = {}
for algorithm_name, params in algorithm_parameters.items():
    run_name = f"original_nfs_{algorithm_name}"
    model, train_clusters, test_clusters, fit_seconds, assignment_metadata = fit_and_assign(
        algorithm_name, params, X_train_cluster_space, X_test_cluster_space
    )
    mapping, default_label, contingency = fit_hungarian_mapping(
        train_clusters, y_cluster_train.to_numpy()
    )
    mapped_test = apply_cluster_mapping(test_clusters, mapping, default_label)
    metrics = all_metrics(
        X_test_cluster_space, test_clusters, y_cluster_test.to_numpy(), mapped_test
    )
    tn, fp, fn, tp = confusion_matrix(y_cluster_test, mapped_test, labels=[0, 1]).ravel()
    metrics.update({
        "mapped_true_negatives": int(tn), "mapped_false_positives": int(fp),
        "mapped_false_negatives": int(fn), "mapped_true_positives": int(tp),
        "fit_seconds": fit_seconds,
    })

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({
            "task": "unsupervised_clustering", "feature_set": "original",
            "data_variant": "Original_NFS", "algorithm": algorithm_name,
            "labels_used_for_training": "false",
            "cluster_mapping": "hungarian_on_training_sample",
        })
        mlflow.log_input(train_dataset, context="clustering_training")
        mlflow.log_input(test_dataset, context="clustering_evaluation")
        mlflow.log_params({
            **params, "algorithm": algorithm_name, "random_state": RANDOM_STATE,
            "original_feature_count": len(ORIGINAL_FEATURES),
            "preprocessed_feature_count": X_train_preprocessed.shape[1],
            "pca_components": X_train_cluster_space.shape[1],
            "pca_explained_variance": pca_cluster.explained_variance_ratio_.sum(),
            "cluster_train_rows": len(X_cluster_train),
            "cluster_test_rows": len(X_cluster_test),
            "full_train_rows": len(X_train), "full_test_rows": len(X_test),
            "numeric_scaling": "StandardScaler",
            "categorical_encoding": "OneHotEncoder(handle_unknown=ignore)",
        })
        mlflow.log_metrics({k: float(v) for k, v in metrics.items() if np.isfinite(v)})
        mlflow.log_dict({
            "cluster_to_binary_label": {str(k): int(v) for k, v in mapping.items()},
            "default_label": default_label, "training_contingency_matrix": contingency,
            **assignment_metadata,
        }, "mapping/cluster_label_mapping.json")
        mlflow.log_dict({
            "original_features": list(ORIGINAL_FEATURES),
            "label_mapping": label_mapping,
            "sampling": {
                "strategy": "stratified_without_replacement",
                "train_rows": len(X_cluster_train), "test_rows": len(X_cluster_test),
                "reason": "common computationally feasible sample for DBSCAN and agglomerative clustering",
            },
            "library_versions": {
                "python": platform.python_version(), "mlflow": mlflow.__version__,
                "scikit_learn": sklearn.__version__,
            },
        }, "metadata/run_metadata.json")

        for plot_name, embedding in {"pca": pca_visual, "tsne": tsne_visual}.items():
            fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
            axes[0].scatter(embedding[:, 0], embedding[:, 1], c=test_clusters[visualization_indices], s=8, cmap="tab10", alpha=0.7)
            axes[0].set_title(f"{algorithm_name}: predicted clusters")
            axes[1].scatter(embedding[:, 0], embedding[:, 1], c=y_cluster_test.to_numpy()[visualization_indices], s=8, cmap="coolwarm", alpha=0.7)
            axes[1].set_title("True labels (evaluation only)")
            for ax in axes: ax.set_xlabel(f"{plot_name.upper()} 1"); ax.set_ylabel(f"{plot_name.upper()} 2")
            fig.tight_layout(); mlflow.log_figure(fig, f"plots/{plot_name}_clusters_vs_labels.png"); plt.close(fig)

        representation_example = X_cluster_train.head(5).copy()
        for column in numeric_features:
            representation_example[column] = representation_example[column].astype("float64")
        mlflow.sklearn.log_model(
            sk_model=representation_pipeline, name="representation_pipeline",
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
            signature=infer_signature(
                representation_example, representation_pipeline.transform(representation_example)
            ), input_example=representation_example,
        )
        mlflow.sklearn.log_model(
            sk_model=model, name="clustering_model",
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        )
        result = {"run_name": run_name, "run_id": run.info.run_id, "algorithm": algorithm_name, **metrics}
        results.append(result); fitted_runs[algorithm_name] = {
            "model": model, "params": params, "train_clusters": train_clusters,
            "test_clusters": test_clusters, "mapping": mapping,
        }
        print(f"Completed {run_name}: silhouette={metrics['silhouette_score']:.4f}, mapped F1={metrics['mapped_f1']:.4f}")

results_df = pd.DataFrame(results).sort_values("mapped_f1", ascending=False).reset_index(drop=True)
results_df

Completed original_nfs_k-means: silhouette=0.4046, mapped F1=0.8976
Completed original_nfs_dbscan: silhouette=0.3162, mapped F1=0.9175
Completed original_nfs_agglomerative-hierarchical: silhouette=0.3942, mapped F1=0.7538
Completed original_nfs_gaussian-mixture-model: silhouette=0.3872, mapped F1=0.2731

Algorithm                     Silhouette   DB Index   CH Index    ARI      NMI      Accuracy   Precision  Recall    F1
DBSCAN                       0.3162       1.6321     181.68      0.4854   0.4353   0.9285     0.9907     0.8545   0.9175
K-Means                      0.4046       1.3266     1097.27     0.6813   0.6258   0.9128     0.9896     0.8212   0.8976
Agglomerative Hierarchical   0.3942       0.8684     1101.74     0.3981   0.4318   0.8158     0.9973     0.6058   0.7538
Gaussian Mixture Model       0.3872       3.4473     212.62      0.0071   0.0042   0.5488     0.5459     0.1821   0.2731


## Hyperparameter tuning of the best initial clustering model

The initial winner is selected by mapped test F1 for practical intrusion classification. Hyperparameters are then tuned using **training silhouette score only**, so test labels do not influence parameter selection. The test set is evaluated once using the winning parameters and the mapping learned from training labels.

In [2]:
best_initial_algorithm = results_df.iloc[0]["algorithm"]
tuning_grids = {
    "k-means": {"n_clusters": [2, 3, 4, 5, 6], "n_init": [20, 40], "max_iter": [500]},
    "dbscan": {"eps": [dbscan_eps * 0.75, dbscan_eps, dbscan_eps * 1.25], "min_samples": [5, 10, 20], "metric": ["euclidean"], "n_jobs": [-1]},
    "agglomerative-hierarchical": {"n_clusters": [2, 3, 4, 5, 6], "linkage": ["ward"]},
    "gaussian-mixture-model": {"n_components": [2, 3, 4, 5, 6], "covariance_type": ["diag", "tied"], "n_init": [5], "max_iter": [300]},
}
tuning_rows = []
for candidate_number, candidate_params in enumerate(ParameterGrid(tuning_grids[best_initial_algorithm]), start=1):
    candidate_model = create_clusterer(best_initial_algorithm, candidate_params)
    started = time.perf_counter()
    if best_initial_algorithm in {"k-means", "gaussian-mixture-model"}:
        candidate_model.fit(X_train_cluster_space)
        candidate_clusters = candidate_model.predict(X_train_cluster_space)
    else:
        candidate_clusters = candidate_model.fit_predict(X_train_cluster_space)
    candidate_internal = safe_internal_metrics(X_train_cluster_space, candidate_clusters)
    tuning_rows.append({
        "candidate": candidate_number, "params": candidate_params,
        **candidate_internal, "fit_seconds": time.perf_counter() - started,
    })
tuning_df = pd.DataFrame(tuning_rows)
valid_tuning = tuning_df.dropna(subset=["silhouette_score"])
best_candidate = valid_tuning.sort_values(
    ["silhouette_score", "davies_bouldin_index"], ascending=[False, True]
).iloc[0]
best_params = best_candidate["params"]
tuned_model, tuned_train_clusters, tuned_test_clusters, tuned_fit_seconds, tuned_assignment = fit_and_assign(
    best_initial_algorithm, best_params, X_train_cluster_space, X_test_cluster_space
)
tuned_mapping, tuned_default, tuned_contingency = fit_hungarian_mapping(
    tuned_train_clusters, y_cluster_train.to_numpy()
)
tuned_mapped_test = apply_cluster_mapping(tuned_test_clusters, tuned_mapping, tuned_default)
tuned_metrics = all_metrics(
    X_test_cluster_space, tuned_test_clusters, y_cluster_test.to_numpy(), tuned_mapped_test
)
tuned_metrics["fit_seconds"] = tuned_fit_seconds
tuned_run_name = f"original_nfs_{best_initial_algorithm}-tuned"
with mlflow.start_run(run_name=tuned_run_name) as tuned_run:
    mlflow.set_tags({
        "task": "unsupervised_clustering", "feature_set": "original",
        "algorithm": best_initial_algorithm, "hyperparameter_tuned": "true",
        "tuning_objective": "training_silhouette_score",
        "labels_used_for_tuning": "false",
    })
    mlflow.log_input(train_dataset, context="clustering_training")
    mlflow.log_input(test_dataset, context="clustering_evaluation")
    mlflow.log_params({
        **best_params, "algorithm": best_initial_algorithm, "random_state": RANDOM_STATE,
        "tuning_candidate_count": len(tuning_df),
        "selection_metric": "training_silhouette_score",
        "cluster_train_rows": len(X_cluster_train),
        "cluster_test_rows": len(X_cluster_test),
        "pca_components": X_train_cluster_space.shape[1],
    })
    mlflow.log_metrics({k: float(v) for k, v in tuned_metrics.items() if np.isfinite(v)})
    mlflow.log_dict({
        "cluster_to_binary_label": {str(k): int(v) for k, v in tuned_mapping.items()},
        "default_label": tuned_default, "training_contingency_matrix": tuned_contingency,
        **tuned_assignment,
    }, "mapping/cluster_label_mapping.json")
    mlflow.log_dict(
        {"candidates": tuning_df.to_dict(orient="records")},
        "tuning/candidate_results.json",
    )
    mlflow.sklearn.log_model(
        sk_model=tuned_model, name="clustering_model",
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
    )
tuned_result_df = pd.DataFrame([{
    "run_name": tuned_run_name, "run_id": tuned_run.info.run_id,
    "algorithm": best_initial_algorithm, **tuned_metrics,
}])
display(tuning_df.sort_values("silhouette_score", ascending=False))
display(tuned_result_df)

Best initial algorithm by mapped F1: DBSCAN
Hyperparameter search: 9 candidates, optimized using training silhouette without labels.

Tuned DBSCAN test results:
Silhouette=0.4993, Davies-Bouldin=1.5275, Calinski-Harabasz=294.98
ARI=0.4954, NMI=0.4483, Accuracy=0.9185, Precision=0.9942, Recall=0.8298, F1=0.9046
Clusters=14, noise ratio=0.0975

Tuning improved geometric separation but reduced mapped F1 from 0.9175 to 0.9046.


In [3]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
tracked_runs = mlflow.search_runs([experiment.experiment_id])
supervised_runs = tracked_runs[
    tracked_runs["tags.mlflow.runName"].fillna("").str.startswith("Original_NFS_")
].sort_values("start_time", ascending=False).drop_duplicates("tags.algorithm")
supervised_comparison = supervised_runs[[
    "tags.mlflow.runName", "tags.algorithm", "metrics.test_accuracy",
    "metrics.test_precision", "metrics.test_recall", "metrics.test_f1",
]].rename(columns={
    "tags.mlflow.runName": "run_name", "tags.algorithm": "algorithm",
    "metrics.test_accuracy": "accuracy",
    "metrics.test_precision": "precision",
    "metrics.test_recall": "recall", "metrics.test_f1": "f1",
})
unsupervised_comparison = pd.concat([
    results_df[["run_name", "algorithm", "mapped_accuracy", "mapped_precision", "mapped_recall", "mapped_f1"]].rename(columns={
        "mapped_accuracy": "accuracy", "mapped_precision": "precision",
        "mapped_recall": "recall", "mapped_f1": "f1",
    }),
    tuned_result_df[["run_name", "algorithm", "mapped_accuracy", "mapped_precision", "mapped_recall", "mapped_f1"]].rename(columns={
        "mapped_accuracy": "accuracy", "mapped_precision": "precision",
        "mapped_recall": "recall", "mapped_f1": "f1",
    }),
])
best_unsupervised = unsupervised_comparison.sort_values("f1", ascending=False).iloc[0]
best_supervised = supervised_comparison.sort_values("f1", ascending=False).iloc[0]
best_internal = results_df.sort_values("silhouette_score", ascending=False).iloc[0]
best_external = results_df.sort_values("mapped_f1", ascending=False).iloc[0]

display(Markdown("## Internal and external clustering comparison"))
display(results_df[[
    "algorithm", "silhouette_score", "davies_bouldin_index",
    "calinski_harabasz_index", "adjusted_rand_index",
    "normalized_mutual_information", "homogeneity", "completeness",
    "v_measure", "mapped_accuracy", "mapped_precision",
    "mapped_recall", "mapped_f1", "cluster_count", "noise_ratio",
]])
display(Markdown("## Supervised versus unsupervised"))
display(pd.concat([
    best_supervised.to_frame().T.assign(learning="supervised"),
    best_unsupervised.to_frame().T.assign(learning="unsupervised"),
]))

separation_strength = (
    "strong" if best_external.mapped_f1 >= 0.90 and best_external.adjusted_rand_index >= 0.60
    else "moderate" if best_external.mapped_f1 >= 0.75 else "weak"
)
display(Markdown(f"""
## Answers to the research questions

### Can the original feature space naturally separate network traffic into meaningful groups?
The evidence indicates **{separation_strength} natural separation**. The best initial mapped test F1 is **{best_external.mapped_f1:.4f}**, while its ARI is **{best_external.adjusted_rand_index:.4f}** and NMI is **{best_external.normalized_mutual_information:.4f}**. Internal geometry should be read alongside these label-based metrics because a geometrically compact partition is not necessarily aligned with Attack/Normal semantics.

### Which clustering algorithm best models the structure?
By mapped test F1, **{best_external.algorithm}** is the strongest initial model ({best_external.mapped_f1:.4f}). By silhouette score, **{best_internal.algorithm}** produces the strongest geometric separation ({best_internal.silhouette_score:.4f}). If these differ, the result demonstrates that geometric cluster quality and intrusion-label alignment are different objectives.

### Are attack classes naturally separable without supervision?
The best clustering model recovers Attack/Normal labels with F1 **{best_external.mapped_f1:.4f}**, compared with **{best_supervised.f1:.4f}** for the strongest supervised original-feature model. The remaining gap quantifies how much label supervision contributes.

### Supervised versus unsupervised conclusion
The best supervised run is `{best_supervised.run_name}` with F1 **{best_supervised.f1:.4f}**. The best unsupervised run is `{best_unsupervised.run_name}` with mapped F1 **{best_unsupervised.f1:.4f}**. Clustering is valuable for structure discovery and anomaly exploration, but the supervised model is preferred for final binary classification whenever its test F1 is materially higher.
"""))

Best supervised: Original_NFS_xgboost -- F1=0.999275
Best unsupervised mapped classifier: original_nfs_dbscan -- F1=0.917532
F1 difference: approximately 0.081743 (8.17 percentage points)

Conclusion: supervised XGBoost is preferred for production classification; clustering is useful for structure discovery and anomaly exploration.


# Measured conclusions

## Results summary

| Algorithm | Silhouette | Davies-Bouldin | Calinski-Harabasz | ARI | NMI | Mapped accuracy | Mapped precision | Mapped recall | Mapped F1 | Clusters / noise |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| K-Means | **0.4046** | 1.3266 | 1097.27 | **0.6813** | **0.6258** | 91.2750% | 98.9644% | 82.1160% | 89.7564% | 2 / 0% |
| DBSCAN | 0.3162 | 1.6321 | 181.68 | 0.4854 | 0.4353 | **92.8500%** | 99.0660% | **85.4458%** | **91.7532%** | 26 / 8.5% |
| Agglomerative | 0.3942 | **0.8684** | **1101.74** | 0.3981 | 0.4318 | 81.5750% | **99.7347%** | 60.5800% | 75.3759% | 2 / 0% |
| Gaussian Mixture | 0.3872 | 3.4473 | 212.62 | 0.0071 | 0.0042 | 54.8750% | 54.5894% | 18.2062% | 27.3057% | 2 / 0% |
| Tuned DBSCAN | 0.4993 | 1.5275 | 294.98 | 0.4954 | 0.4483 | 91.8500% | 99.4208% | 82.9753% | 90.4567% | 14 / 9.75% |

## Can the original feature space naturally separate traffic into meaningful groups?

**Yes, but only moderately without supervision.** K-Means recovers a meaningful two-group structure with ARI 0.6813, NMI 0.6258, and silhouette 0.4046. DBSCAN reaches 91.7532% mapped F1, confirming substantial Attack/Normal signal, but it forms 26 clusters and marks 8.5% of test samples as noise. The data therefore contain natural structure, although it is not a clean one-to-one binary partition.

## Which clustering algorithm best models network-traffic structure?

**K-Means is the best model of the underlying two-class structure.** It has the highest initial silhouette, ARI, NMI, homogeneity/completeness balance, and a natural two-cluster output. **DBSCAN is the best unsupervised classifier after label mapping**, with the highest mapped accuracy, recall, and F1, but its 26 micro-clusters and noise group make it less direct as a model of binary structure. Agglomerative clustering has the best Davies-Bouldin and Calinski-Harabasz values, but its much lower recall and F1 show that compact geometry does not align as closely with Attack/Normal labels.

## Are attack classes naturally separable without supervision?

They are **partially separable**. DBSCAN detects attacks with 85.4458% recall and 91.7532% mapped F1; K-Means produces stronger cluster/label agreement but lower recall. This is useful for exploratory grouping and anomaly discovery, but it leaves a meaningful classification gap.

## Hyperparameter-tuning result

DBSCAN was tuned because it had the highest initial mapped F1. Tuning without labels increased silhouette from 0.3162 to **0.4993** and reduced the number of discovered groups from 26 to 14. However, mapped F1 fell from 91.7532% to **90.4567%**. This demonstrates that optimizing geometric separation does not necessarily optimize alignment with intrusion labels. Keep the original DBSCAN parameters for mapped classification; use tuned DBSCAN when cleaner geometric clusters are the priority.

## Supervised versus unsupervised

The best supervised original-feature model, `Original_NFS_xgboost`, achieved **99.9275% F1**, compared with **91.7532%** for the best unsupervised mapped model, `original_nfs_dbscan`—an F1 gap of approximately **8.17 percentage points**. Supervised XGBoost is therefore the correct choice for final Attack/Normal classification. Unsupervised clustering remains valuable for discovering subgroups, identifying noise/outliers, and investigating previously unseen attack behavior.

> **Final decision:** Use **K-Means** when the goal is to describe the natural two-group structure, use the original **DBSCAN** when maximizing mapped unsupervised classification F1, and use supervised **XGBoost** for production binary intrusion classification.

# Compare algorithms directly from MLflow

The `algorithm` value is logged as both a tag and a parameter. For Silhouette, Calinski-Harabasz, ARI, NMI, Homogeneity, Completeness, V-measure, Accuracy, Precision, Recall, and F1, **higher is better**. For the Davies-Bouldin Index, **lower is better**. The following cell loads the latest version of each unsupervised run and creates a comparison table plus a winner for every metric.

In [4]:
comparison_experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
comparison_runs = mlflow.search_runs([comparison_experiment.experiment_id])
comparison_runs = comparison_runs[
    comparison_runs["tags.task"].eq("unsupervised_clustering")
].sort_values("start_time", ascending=False).drop_duplicates("tags.mlflow.runName")
comparison_runs["algorithm_display"] = comparison_runs["params.algorithm"].fillna(
    comparison_runs["tags.algorithm"]
)
mlflow_algorithm_comparison_df = comparison_runs[[
    "tags.mlflow.runName", "algorithm_display",
    "metrics.silhouette_score", "metrics.davies_bouldin_index",
    "metrics.calinski_harabasz_index", "metrics.adjusted_rand_index",
    "metrics.normalized_mutual_information", "metrics.homogeneity",
    "metrics.completeness", "metrics.v_measure",
    "metrics.mapped_accuracy", "metrics.mapped_precision",
    "metrics.mapped_recall", "metrics.mapped_f1",
    "metrics.cluster_count", "metrics.noise_ratio",
]].rename(columns={
    "tags.mlflow.runName": "run_name", "algorithm_display": "algorithm",
    **{column: column.replace("metrics.", "") for column in comparison_runs.columns if column.startswith("metrics.")},
}).sort_values("mapped_f1", ascending=False).reset_index(drop=True)

higher_is_better = [
    "silhouette_score", "calinski_harabasz_index", "adjusted_rand_index",
    "normalized_mutual_information", "homogeneity", "completeness",
    "v_measure", "mapped_accuracy", "mapped_precision",
    "mapped_recall", "mapped_f1",
]
winner_rows = []
for metric in higher_is_better:
    winner_index = mlflow_algorithm_comparison_df[metric].idxmax()
    winner_rows.append({
        "metric": metric,
        "direction": "higher is better",
        "winning_run": mlflow_algorithm_comparison_df.loc[winner_index, "run_name"],
        "value": mlflow_algorithm_comparison_df.loc[winner_index, metric],
    })
db_winner_index = mlflow_algorithm_comparison_df["davies_bouldin_index"].idxmin()
winner_rows.append({
    "metric": "davies_bouldin_index", "direction": "lower is better",
    "winning_run": mlflow_algorithm_comparison_df.loc[db_winner_index, "run_name"],
    "value": mlflow_algorithm_comparison_df.loc[db_winner_index, "davies_bouldin_index"],
})
metric_winners_df = pd.DataFrame(winner_rows)
display(mlflow_algorithm_comparison_df)
display(Markdown("## Winner for each metric"))
display(metric_winners_df)

MLflow algorithm comparison (latest runs)

Run                                      Algorithm                    Silhouette   ARI      NMI      Accuracy   Precision  Recall    F1
original_nfs_dbscan                      dbscan                       0.3162       0.4854   0.4353   0.9285     0.9907     0.8545   0.9175
original_nfs_dbscan-tuned                dbscan                       0.4993       0.4954   0.4483   0.9185     0.9942     0.8298   0.9046
original_nfs_k-means                     k-means                      0.4046       0.6813   0.6258   0.9128     0.9896     0.8212   0.8976
original_nfs_agglomerative-hierarchical agglomerative-hierarchical  0.3942       0.3981   0.4318   0.8158     0.9973     0.6058   0.7538
original_nfs_gaussian-mixture-model     gaussian-mixture-model      0.3872       0.0071   0.0042   0.5488     0.5459     0.1821   0.2731

Metric winners: silhouette=tuned DBSCAN; ARI/NMI=K-Means; DB index/CH index=Agglomerative; mapped accuracy/recall/F1=original DBS

# Attack-family and rare-attack diagnostics for DBSCAN

DBSCAN is analyzed because it achieved the highest mapped unsupervised F1. Binary cluster-to-label mapping measures whether each attack is detected at all. A separate training-only majority-vote mapping from clusters to `attack_category` measures family-level precision, recall, and F1. Neither mapping changes the clustering model. Feature importance is obtained from a Random Forest surrogate trained to reproduce DBSCAN cluster assignments; it describes features associated with cluster formation, not causal attack predictors.

In [5]:
diagnostic_algorithm = "dbscan"
diagnostic_record = fitted_runs[diagnostic_algorithm]
train_clusters = diagnostic_record["train_clusters"]
test_clusters = diagnostic_record["test_clusters"]
attack_family_train = df.loc[X_cluster_train.index, "attack_category"].astype(str).to_numpy()
attack_family_test = df.loc[X_cluster_test.index, "attack_category"].astype(str).to_numpy()
attack_type_test = df.loc[X_cluster_test.index, "attack"].astype(str).to_numpy()
family_order = ["Normal", "DoS", "Probe", "R2L", "U2R"]

def fit_majority_cluster_mapping(cluster_ids, labels):
    mapping = {}
    global_majority = str(pd.Series(labels).mode().iloc[0])
    for cluster_id in np.unique(cluster_ids):
        members = labels[cluster_ids == cluster_id]
        mapping[int(cluster_id)] = str(pd.Series(members).mode().iloc[0])
    return mapping, global_majority

family_mapping, default_family = fit_majority_cluster_mapping(
    train_clusters, attack_family_train
)
predicted_families = np.array([
    family_mapping.get(int(cluster_id), default_family) for cluster_id in test_clusters
])
family_precision, family_recall, family_f1, family_support = precision_recall_fscore_support(
    attack_family_test, predicted_families, labels=family_order, zero_division=0
)
family_metrics_df = pd.DataFrame({
    "attack_family": family_order, "support": family_support,
    "precision": family_precision, "recall": family_recall, "f1": family_f1,
})

family_confusion = confusion_matrix(
    attack_family_test, predicted_families, labels=family_order
)
one_vs_rest_rows = []
for family in family_order:
    true_binary = attack_family_test == family
    pred_binary = predicted_families == family
    tn, fp, fn, tp = confusion_matrix(true_binary, pred_binary, labels=[False, True]).ravel()
    one_vs_rest_rows.append({
        "attack_family": family, "tn": int(tn), "fp": int(fp),
        "fn": int(fn), "tp": int(tp),
    })
attack_wise_confusion_df = pd.DataFrame(one_vs_rest_rows)

binary_mapping, binary_default, _ = fit_hungarian_mapping(
    train_clusters, y_cluster_train.to_numpy()
)
binary_attack_predictions = apply_cluster_mapping(test_clusters, binary_mapping, binary_default)
detection_frame = pd.DataFrame({
    "attack_family": attack_family_test, "attack_type": attack_type_test,
    "detected_as_attack": binary_attack_predictions == 1,
})
family_detection_rate_df = (
    detection_frame.groupby("attack_family")["detected_as_attack"]
    .agg(["count", "sum", "mean"]).reset_index()
    .rename(columns={"count": "support", "sum": "detected", "mean": "detection_rate"})
)
attack_detection_rate_df = (
    detection_frame[detection_frame.attack_family != "Normal"]
    .groupby(["attack_family", "attack_type"])["detected_as_attack"]
    .agg(["count", "sum", "mean"]).reset_index()
    .rename(columns={"count": "support", "sum": "detected", "mean": "detection_rate"})
    .sort_values(["detection_rate", "support"], ascending=[False, False])
)

cluster_family_counts = pd.crosstab(
    pd.Series(test_clusters, name="cluster"),
    pd.Series(attack_family_test, name="attack_family"),
).reindex(columns=family_order, fill_value=0)
cluster_family_percent = cluster_family_counts.div(cluster_family_counts.sum(axis=1), axis=0)
cluster_mixture_df = cluster_family_counts.copy()
cluster_mixture_df["total"] = cluster_family_counts.sum(axis=1)
cluster_mixture_df["dominant_family"] = cluster_family_counts.idxmax(axis=1)
cluster_mixture_df["purity"] = cluster_family_percent.max(axis=1)
cluster_mixture_df["entropy"] = cluster_family_percent.apply(
    lambda row: float(-(row[row > 0] * np.log2(row[row > 0])).sum()), axis=1
)
cluster_mixture_df = cluster_mixture_df.reset_index().sort_values("total", ascending=False)

surrogate_indices = np.arange(len(X_test_preprocessed))
surrogate_train_idx, surrogate_valid_idx = train_test_split(
    surrogate_indices, test_size=0.25, random_state=RANDOM_STATE
)
surrogate = RandomForestClassifier(
    n_estimators=300, class_weight="balanced_subsample",
    random_state=RANDOM_STATE, n_jobs=-1,
)
surrogate.fit(X_test_preprocessed[surrogate_train_idx], test_clusters[surrogate_train_idx])
surrogate_fidelity = accuracy_score(
    test_clusters[surrogate_valid_idx], surrogate.predict(X_test_preprocessed[surrogate_valid_idx])
)
surrogate.fit(X_test_preprocessed, test_clusters)
encoded_raw_names = list(numeric_features)
onehot = preprocessor.named_transformers_["categorical"].named_steps["encoder"]
for raw_feature, categories in zip(categorical_features, onehot.categories_):
    encoded_raw_names.extend([raw_feature] * len(categories))
assert len(encoded_raw_names) == len(surrogate.feature_importances_)
cluster_feature_importance_df = (
    pd.DataFrame({"feature": encoded_raw_names, "importance": surrogate.feature_importances_})
    .groupby("feature", as_index=False).importance.sum()
    .sort_values("importance", ascending=False).reset_index(drop=True)
)

rare_family_threshold = max(5, int(0.01 * len(attack_family_test)))
rare_family_analysis_df = family_detection_rate_df[
    family_detection_rate_df.support <= rare_family_threshold
].merge(family_metrics_df, on=["attack_family", "support"], how="left")
rare_attack_threshold = max(5, int(0.005 * len(attack_family_test)))
rare_attack_analysis_df = attack_detection_rate_df[
    attack_detection_rate_df.support <= rare_attack_threshold
].copy()

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(
    confusion_matrix=family_confusion, display_labels=family_order
).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=30)
ax.set_title("DBSCAN clusters mapped to attack families")
fig.tight_layout()
attack_family_confusion_figure = fig

fig, ax = plt.subplots(figsize=(9, 5))
family_plot = family_metrics_df.set_index("attack_family")[["precision", "recall", "f1"]]
family_plot.plot(kind="bar", ylim=(0, 1.05), ax=ax)
ax.set_title("Attack-family precision, recall, and F1")
ax.set_ylabel("Score"); ax.tick_params(axis="x", rotation=30); fig.tight_layout()
attack_family_metrics_figure = fig

fig, ax = plt.subplots(figsize=(12, 5))
attack_plot = attack_detection_rate_df.sort_values("detection_rate", ascending=False)
bars = ax.bar(attack_plot.attack_type, attack_plot.detection_rate, color="#d95f02")
ax.set_ylim(0, 1.08); ax.set_ylabel("Detection rate"); ax.set_title("DBSCAN binary detection rate by attack type")
ax.tick_params(axis="x", rotation=55)
for bar, support in zip(bars, attack_plot.support):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015, f"n={support}", ha="center", va="bottom", fontsize=8, rotation=90)
fig.tight_layout(); attack_detection_figure = fig

fig, ax = plt.subplots(figsize=(10, 6))
top_importance = cluster_feature_importance_df.head(20).sort_values("importance")
ax.barh(top_importance.feature, top_importance.importance, color="#1b9e77")
ax.set_title(f"Features driving DBSCAN cluster assignments (surrogate fidelity={surrogate_fidelity:.3f})")
ax.set_xlabel("Aggregated Random Forest importance"); fig.tight_layout()
cluster_feature_importance_figure = fig

fig, ax = plt.subplots(figsize=(14, 6))
cluster_family_percent.loc[cluster_mixture_df.cluster].plot(
    kind="bar", stacked=True, ax=ax, colormap="tab20"
)
ax.set_ylabel("Attack-family proportion"); ax.set_title("Attack distribution inside each DBSCAN cluster")
ax.legend(title="Family", bbox_to_anchor=(1.01, 1), loc="upper left"); fig.tight_layout()
cluster_attack_distribution_figure = fig

dbscan_run = mlflow.search_runs(
    [mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id]
).query("`tags.mlflow.runName` == 'original_nfs_dbscan'").sort_values("start_time", ascending=False).iloc[0]
with mlflow.start_run(run_id=dbscan_run.run_id):
    mlflow.set_tags({
        "attack_family_evaluation": "true",
        "attack_family_mapping": "training_cluster_majority_vote",
        "cluster_feature_importance": "random_forest_surrogate",
    })
    attack_metrics_to_log = {"cluster_surrogate_fidelity": surrogate_fidelity}
    for row in family_metrics_df.itertuples(index=False):
        slug = row.attack_family.lower().replace("-", "_")
        attack_metrics_to_log.update({
            f"family_{slug}_precision": float(row.precision),
            f"family_{slug}_recall": float(row.recall),
            f"family_{slug}_f1": float(row.f1),
        })
    for row in family_detection_rate_df.itertuples(index=False):
        slug = row.attack_family.lower().replace("-", "_")
        attack_metrics_to_log[f"family_{slug}_detection_rate"] = float(row.detection_rate)
    mlflow.log_metrics(attack_metrics_to_log)
    mlflow.log_table(family_metrics_df, "attack_diagnostics/family_metrics.json")
    mlflow.log_table(attack_wise_confusion_df, "attack_diagnostics/one_vs_rest_confusion.json")
    mlflow.log_table(family_detection_rate_df, "attack_diagnostics/family_detection_rates.json")
    mlflow.log_table(attack_detection_rate_df, "attack_diagnostics/attack_type_detection_rates.json")
    mlflow.log_table(cluster_mixture_df, "attack_diagnostics/cluster_family_mixture.json")
    mlflow.log_table(cluster_feature_importance_df, "attack_diagnostics/cluster_feature_importance.json")
    mlflow.log_table(rare_family_analysis_df, "attack_diagnostics/rare_family_analysis.json")
    mlflow.log_table(rare_attack_analysis_df, "attack_diagnostics/rare_attack_analysis.json")
    mlflow.log_dict({
        "cluster_to_attack_family": {str(k): v for k, v in family_mapping.items()},
        "default_family": default_family,
        "rare_family_threshold": rare_family_threshold,
        "rare_attack_threshold": rare_attack_threshold,
        "surrogate_fidelity": surrogate_fidelity,
    }, "attack_diagnostics/methodology.json")
    mlflow.log_figure(attack_family_confusion_figure, "attack_diagnostics/attack_family_confusion_matrix.png")
    mlflow.log_figure(attack_family_metrics_figure, "attack_diagnostics/attack_family_metrics.png")
    mlflow.log_figure(attack_detection_figure, "attack_diagnostics/attack_type_detection_rate.png")
    mlflow.log_figure(cluster_feature_importance_figure, "attack_diagnostics/cluster_feature_importance.png")
    mlflow.log_figure(cluster_attack_distribution_figure, "attack_diagnostics/cluster_attack_distribution.png")

display(Markdown("## Precision, recall, and F1 by attack family")); display(family_metrics_df)
display(Markdown("## One-vs-rest confusion counts")); display(attack_wise_confusion_df)
display(Markdown("## Binary detection rate by family")); display(family_detection_rate_df)
display(Markdown("## Binary detection rate by named attack")); display(attack_detection_rate_df)
display(Markdown("## Attack distribution and purity inside each cluster")); display(cluster_mixture_df)
display(Markdown("## Features associated with cluster assignments")); display(cluster_feature_importance_df.head(20))
display(Markdown("## Rare attack analysis")); display(rare_family_analysis_df); display(rare_attack_analysis_df)
display(attack_family_confusion_figure)
display(attack_family_metrics_figure)
display(attack_detection_figure)
display(cluster_attack_distribution_figure)
display(cluster_feature_importance_figure)

Attack-family metrics for original_nfs_dbscan

Family   Support   Precision   Recall   F1       TN    FP   FN   TP   Binary detection rate
Normal   2138      0.8868      0.9930   0.9369   1591  271   15   2123 0.0070 false-alarm rate
DoS      1476      1.0000      0.9146   0.9554   2524    0  126   1350 0.9146
Probe     361      0.9712      0.6537   0.7815   3632    7  125    236 0.6537
R2L        23      0.3846      0.2174   0.2778   3969    8   18      5 0.2174
U2R         2      0.0000      0.0000   0.0000   3998    0    2      0 0.0000

Named attack detection rates:
neptune=99.48% (1328/1335), ipsweep=89.52% (94/105), nmap=77.78% (42/54), teardrop=73.33% (22/30), portsweep=50.00% (51/102), satan=49.00% (49/100), warezclient=21.74% (5/23), smurf=0% (0/86), back=0% (0/19), pod=0% (0/6), loadmodule=0% (0/1), rootkit=0% (0/1).

Top cluster-driving features: service, dsthostsrvdiffhostrate, dsthostdiffsrvrate, flag, dsthostsrvcount, srcbytes, dsthostsamesrvrate, diffsrvrate, srvcount, s

# Attack-wise conclusions

## Diagnostic plots

### Attack-family confusion matrix
![Attack-family confusion matrix](mlruns/1/a66b24dc0819471c97ce2c21b04ffc4a/artifacts/attack_diagnostics/attack_family_confusion_matrix.png)

### Precision, recall, and F1 by attack family
![Attack-family metrics](mlruns/1/a66b24dc0819471c97ce2c21b04ffc4a/artifacts/attack_diagnostics/attack_family_metrics.png)

### Detection rate by named attack
![Attack detection rates](mlruns/1/a66b24dc0819471c97ce2c21b04ffc4a/artifacts/attack_diagnostics/attack_type_detection_rate.png)

### Attack distribution inside each DBSCAN cluster
![Cluster attack distribution](mlruns/1/a66b24dc0819471c97ce2c21b04ffc4a/artifacts/attack_diagnostics/cluster_attack_distribution.png)

### Features associated with cluster formation
![Cluster feature importance](mlruns/1/a66b24dc0819471c97ce2c21b04ffc4a/artifacts/attack_diagnostics/cluster_feature_importance.png)

## Which attacks are easy?

**DoS is the easiest family**, with precision 1.0000, recall 0.9146, and F1 0.9554. At the named-attack level, Neptune is easiest (99.48% detection), followed by Ipsweep (89.52%), Nmap (77.78%), and Teardrop (73.33%). These attacks create strong traffic-volume, service, flag, error-rate, and host-pattern signatures that DBSCAN can isolate.

## Which attacks are difficult?

**U2R is hardest** (0 precision/recall/F1; 0/2 detected), followed by **R2L** (precision 0.3846, recall 0.2174, F1 0.2778). Probe is only moderately separable (F1 0.7815), with Portsweep at 50% and Satan at 49% detection. Smurf, Back, Pod, Loadmodule, and Rootkit are completely missed in this sample. U2R has only two test examples, so its estimate is highly uncertain, but the failure is consistent with rare attacks being absorbed into larger normal or noise regions.

## Where does DBSCAN fail?

DBSCAN fails when attacks resemble normal sessions, occur too rarely to form a dense region, or represent distinct subtypes hidden inside a broad family. It misses 126 DoS, 125 Probe, 18 R2L, and both U2R examples in the family mapping. Cluster 0 mixes 1,783 Normal, 102 DoS, 46 Probe, and 14 R2L records; the noise cluster mixes all five families and has only 68.82% purity. R2L examples also appear in a small cluster dominated by Normal traffic. These mixtures explain the poor rare-attack recall.

## Do clusters correspond to one family or mixtures?

Many small DBSCAN clusters are pure: large clusters 2 and 4 are 100% DoS, while clusters 1, 5, and 7 are 100% Probe. However, the largest cluster is 91.67% Normal and contains multiple attack families, and the noise cluster is strongly mixed. Therefore DBSCAN discovers several meaningful attack-specific micro-clusters, but it does not produce a clean global partition of attack families.

## Cluster feature importance

A Random Forest surrogate identifies `service`, `dsthostsrvdiffhostrate`, `dsthostdiffsrvrate`, `flag`, `dsthostsrvcount`, `srcbytes`, `dsthostsamesrvrate`, `diffsrvrate`, `srvcount`, and `samesrvrate` as the strongest variables associated with DBSCAN assignments. These are cluster-explanation features, not causal importances; the surrogate fidelity metric is logged in MLflow and should be checked before relying on the ranking.

## Rare-attack inference

R2L has only 23 test examples and 21.74% detection. U2R has two examples and neither is detected. Rare named attacks Back (19), Pod (6), Loadmodule (1), and Rootkit (1) are all missed. Density-based clustering is structurally disadvantaged here because rare attacks cannot form stable dense neighborhoods. For rare-attack detection, use supervised class weighting/resampling, anomaly-detection methods trained around normal traffic, or a hierarchical pipeline that first detects Attack versus Normal and then classifies the attack family.

> **Overall inference:** DBSCAN is strong for dominant DoS patterns and several Probe subtypes, but it is unsuitable as the sole detector for rare R2L/U2R attacks. The most important operational failures are Smurf, Back, Pod, Warezclient, Loadmodule, and Rootkit.